[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivamkalakaar/local-ai-lab-agents/blob/main/cs329a/a05_rl_training.ipynb)

# a05 · Train-time scaling: STaR, GRPO and DAPO on a real (tiny) model
**Local AI Lab** · CS329A Lecture 6 · make a model better by training on its own checked answers

So far we spent more compute at answer time. Here we spend it at training time. The model tries problems, a checker grades the answers, and we train on what worked.
You will (1) build a STaR dataset with a local Llama model, (2) compute GRPO advantages by hand, (3) see what DAPO changes, and (4) run a real GRPO training run with LoRA on SmolLM2-360M, with accuracy before and after.
The whole notebook takes about 15 minutes on a free Colab T4. The training run is about 8-10 minutes of that.

In [ ]:
# Local AI Lab setup: run this cell first (about 2-3 minutes on Colab, a few seconds locally)
import os, sys, subprocess
EXTRA = ['trl', 'peft', 'datasets']
if "google.colab" in sys.modules:
    if not os.path.exists("/content/local-ai-lab-agents"):
        subprocess.run(["git", "clone", "-q", "https://github.com/shivamkalakaar/local-ai-lab-agents", "/content/local-ai-lab-agents"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/local-ai-lab-agents", "pull", "-q"])   # get the latest fixes
    os.chdir("/content/local-ai-lab-agents")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", *EXTRA], check=True)
elif os.path.basename(os.getcwd()) in ("notebooks", "projects", "cs329a", "cs329a_projects"):
    os.chdir("..")
sys.path.insert(0, os.getcwd())
from lal.runtime import setup
MODEL = setup(['llama3.2:3b'])

In [ ]:
import json, random, re, time, urllib.request
from concurrent.futures import ThreadPoolExecutor
import numpy as np
from lal.llm import ask
from lal.checks import check

STUDENT = "llama3.2:3b"     # a small Ollama model that still makes mistakes, so STaR has something to fix
RUN_LONG = False            # True = a longer GRPO run (about 30-40 minutes on a T4)

## 1. A task with exact answers
RL from rewards only works when a program can grade the answer. We generate arithmetic problems with a fixed seed, so every run sees the same problems and every answer is either right or wrong.
The model must end with a line `Answer: <number>`. We read that line with a regex. If it is missing we fall back to the last number in the text.

In [ ]:
def make_problems(n, seed, hard=False):
    """Seeded arithmetic problems. hard=True gives 3-step problems that trip up small models."""
    rng, out = random.Random(seed), []
    for _ in range(n):
        if hard:
            a, b, c, d = rng.randint(12, 49), rng.randint(3, 9), rng.randint(10, 99), rng.randint(10, 99)
            q, ans = f"What is {a} * {b} - {c} + {d}?", a * b - c + d
        else:
            op = rng.choice(["+", "-", "*"])
            if op == "*":
                a, b = rng.randint(3, 9), rng.randint(11, 39)
            else:
                a, b = rng.randint(10, 99), rng.randint(10, 99)
            q, ans = f"What is {a} {op} {b}?", eval(f"{a}{op}{b}")       # safe: we built the string ourselves
        out.append({"question": q, "answer": int(ans)})
    return out

def extract_int(text):
    """Number after 'Answer:' if present, otherwise the last integer in the text. None if no number."""
    text = text or ""
    m = re.findall(r"Answer:\s*\**\s*(-?\d[\d,]*)", text, re.I)
    if not m:
        m = re.findall(r"-?\d[\d,]*", text)
    if not m:
        return None
    try:
        return int(m[-1].replace(",", ""))
    except ValueError:
        return None

def has_answer_line(text):
    return bool(re.search(r"Answer:\s*\**\s*-?\d", text or "", re.I))

INSTR = "Think step by step in one or two short lines, then finish with the line 'Answer: <number>'."
print(make_problems(3, seed=0), extract_int("so 12*3=36. Answer: 36"), extract_int("I think 41"))

## 2. STaR: the model writes its own training data
STaR (Zelikman et al. 2022, *STaR: Bootstrapping Reasoning With Reasoning*) is a simple loop:
1. Ask the model to reason and answer many problems.
2. Keep the reasoning only when the final answer is right.
3. For the problems it got wrong, show it the correct answer as a **hint** and ask it to explain. This is called **rationalization**. If the explanation now ends at the right answer, keep it, but **store it without the hint**, as if the model had found it alone.
4. Fine-tune on the kept examples and repeat.

In the paper, GPT-J (6B) with STaR reached 72.5% on CommonsenseQA, close to 73.0% for a fine-tuned GPT-3 that is 30 times larger.
Here we do steps 1-3 with the 3B Ollama model and save the dataset. Step 4 is the same supervised fine-tuning you would do with any chat dataset.

In [ ]:
star_problems = make_problems(16, seed=1, hard=True)

def rationale(p, hint=None):
    prompt = f"{p['question']}\n{INSTR}"
    if hint is not None:
        prompt += f"\n(Hint: the correct answer is {hint}. Explain the steps that lead to it.)"
    try:
        return ask(prompt, model_name=STUDENT, temperature=0.7, max_tokens=200)
    except Exception as e:
        return f"(error: {e})"

t0 = time.time()
with ThreadPoolExecutor(2) as ex:
    first = list(ex.map(rationale, star_problems))                       # 16 calls
wrong = [i for i, (p, r) in enumerate(zip(star_problems, first)) if extract_int(r) != p["answer"]]
with ThreadPoolExecutor(2) as ex:
    hinted = dict(zip(wrong, ex.map(lambda i: rationale(star_problems[i], hint=star_problems[i]["answer"]), wrong)))  # <= 16 calls

sft = []
for i, (p, r) in enumerate(zip(star_problems, first)):
    if i not in wrong:
        sft.append({"source": "self", "messages": [{"role": "user", "content": f"{p['question']}\n{INSTR}"},
                                                   {"role": "assistant", "content": r.strip()}]})
    elif extract_int(hinted[i]) == p["answer"] and "hint" not in hinted[i].lower():
        # the prompt we store has NO hint: the model learns to produce this reasoning on its own
        sft.append({"source": "rationalized", "messages": [{"role": "user", "content": f"{p['question']}\n{INSTR}"},
                                                           {"role": "assistant", "content": hinted[i].strip()}]})
with open("star_sft.jsonl", "w") as f:
    for row in sft:
        f.write(json.dumps(row) + "\n")

n_self = sum(r["source"] == "self" for r in sft)
print(f"{len(star_problems)} problems: {len(star_problems) - len(wrong)} right on the first try, {len(wrong)} wrong")
print(f"kept {n_self} self-made + {len(sft) - n_self} rationalized examples -> star_sft.jsonl  [{time.time() - t0:.0f}s]")
if sft:
    print("\nexample:", sft[0]["messages"][1]["content"][:300])

Two things to notice:
- **The checker is the teacher.** No human wrote any of these explanations. We only kept the ones whose final number was right.
- **A right answer can still have a wrong explanation.** Rationalized examples are the riskiest: the model knew the answer and may have bent the steps to fit. We drop any reply that mentions the hint, and in real use you would also check the steps (for arithmetic you can recompute each line).

## 3. GRPO by hand: grade against the group, not against a critic
GRPO (Shao et al. 2024, *DeepSeekMath*) is the RL method behind many recent reasoning models. For each prompt:
1. Sample a **group** of G answers (the paper used G = 64).
2. Score each one with the reward.
3. The **advantage** of an answer is how much better it did than its group: `(reward - group mean) / group std`.
4. Raise the probability of tokens in answers with positive advantage, lower it for negative ones.

PPO needs a second network (a "critic" or value model) to guess the expected reward. GRPO uses the group mean instead, which saves a lot of memory. In the paper, GRPO lifted DeepSeekMath-Instruct 7B from 82.9% to 88.2% on GSM8K and from 46.8% to 51.7% on MATH.

Let us get real rewards: 4 samples each for 3 problems from the 3B model (12 calls).

In [ ]:
REWARD_CORRECT, REWARD_FORMAT = 1.0, 0.2

def score(text, gold):
    return (REWARD_CORRECT if extract_int(text) == gold else 0.0) + (REWARD_FORMAT if has_answer_line(text) else 0.0)

G = 4
group_problems = make_problems(3, seed=2, hard=True)
jobs = [p for p in group_problems for _ in range(G)]
with ThreadPoolExecutor(2) as ex:
    outs = list(ex.map(rationale, jobs))
groups = [np.array([score(o, p["answer"]) for o, p in zip(outs[k * G:(k + 1) * G], jobs[k * G:(k + 1) * G])])
          for k in range(len(group_problems))]

def grpo_advantages(r, eps=1e-4):
    r = np.asarray(r, dtype=float)
    return (r - r.mean()) / (r.std() + eps)

for p, r in zip(group_problems, groups):
    print(f"{p['question']:<32} rewards {r}  ->  advantages {grpo_advantages(r).round(2)}")

Read the output like this: a group where every answer got the same reward has advantage 0 for all of them. That prompt teaches the model **nothing**, even though we paid to generate it. DAPO fixes exactly this.

## 4. What DAPO changes (Yu et al. 2025, *DAPO*)
DAPO starts from GRPO and adds four fixes. In their ablation on AIME 2024 with Qwen2.5-32B, plain GRPO scored 30, and the full DAPO recipe scored 50. That beat DeepSeek-R1-Zero-Qwen-32B (47) with 50% of its training steps. DAPO also drops the KL penalty.

| Fix | Problem it solves | In TRL's `GRPOConfig` |
|---|---|---|
| **Clip-higher**: clip the probability ratio at 1-0.2 below but 1+0.28 above | With symmetric clipping, rare but good tokens cannot grow fast, so the model stops exploring ("entropy collapse") | `epsilon=0.2, epsilon_high=0.28` |
| **Dynamic sampling**: drop groups where all rewards are equal, sample more until the batch is full | Those groups give zero gradient and waste the batch | not built in; we measure it below |
| **Token-level loss**: average the loss over all tokens in the batch, not per answer first | Per-answer averaging lets a long bad answer count the same as a short one | `loss_type="dapo"` (TRL's default now) |
| **Overlong reward shaping**: soft penalty that grows as an answer nears the length limit | Cut-off answers get a reward of 0 for the wrong reason, which adds noise | a reward function, below; or `mask_truncated_completions=True` |

Here is dynamic sampling on the rewards we just collected, and the soft overlong penalty as a function.

In [ ]:
def dynamic_sampling(groups):
    """Keep only groups whose rewards are not all equal (DAPO)."""
    return [g for g in groups if np.ptp(g) > 0]

def overlong_penalty(n_tokens, l_max=96, l_cache=32):
    """DAPO soft overlong punishment: 0 until l_max - l_cache tokens, then down to -1 at l_max."""
    if n_tokens <= l_max - l_cache:
        return 0.0
    if n_tokens <= l_max:
        return (l_max - l_cache - n_tokens) / l_cache
    return -1.0

kept = dynamic_sampling(groups)
print(f"dynamic sampling keeps {len(kept)} of {len(groups)} groups")
print("overlong penalty at 40, 70, 90, 96 tokens:", [round(overlong_penalty(n), 2) for n in (40, 70, 90, 96)])

## 5. A real GRPO run: SmolLM2-360M + LoRA on a T4
Now we train for real with Hugging Face TRL's `GRPOTrainer`.
- **Model:** `HuggingFaceTB/SmolLM2-360M-Instruct`, a small open model. We train only a **LoRA** adapter (low-rank extra weights, about 1% of the model), so memory stays small.
- **Rewards:** +1.0 for the exact answer, +0.2 for the `Answer:` line, and DAPO's soft overlong penalty (scaled by 0.5).
- **Group size 8**, completions of at most 96 tokens, 120 steps of 2 prompts each.
- **Precision:** the T4 has no bfloat16, so we keep the weights in float32 and use float16 mixed precision. TRL's config turns `bf16` on by default, so we switch it off explicitly.

First we free the GPU: Ollama keeps the Llama models loaded, and we no longer need them.

In [ ]:
def unload_ollama(models):
    for m in models:
        try:
            req = urllib.request.Request("http://localhost:11434/api/generate",
                                         data=json.dumps({"model": m, "keep_alive": 0}).encode(),
                                         headers={"Content-Type": "application/json"})
            urllib.request.urlopen(req, timeout=30).read()
        except Exception as e:
            print("could not unload", m, e)
unload_ollama([MODEL, STUDENT])

import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer

HAS_GPU = torch.cuda.is_available()
BF16_OK = HAS_GPU and torch.cuda.get_device_capability()[0] >= 8     # Ampere or newer. T4 is 7.5 -> False
MAX_STEPS = (400 if RUN_LONG else 120) if HAS_GPU else 5              # on CPU, just prove the code runs
MAX_NEW = 96
MID = "HuggingFaceTB/SmolLM2-360M-Instruct"

tok = AutoTokenizer.from_pretrained(MID)
tok.padding_side = "left"                     # GRPO and batched generation need left padding
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
policy = AutoModelForCausalLM.from_pretrained(MID).float()       # float32 weights; .float() also undoes any bf16 default
if HAS_GPU:
    policy = policy.to("cuda")
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "none", "| bf16 supported:", BF16_OK, "| steps:", MAX_STEPS)

def to_prompt(q):
    return [{"role": "user", "content": f"{q}\n{INSTR}"}]

train_rows = make_problems(MAX_STEPS * 2 + 50, seed=100)
test_rows = make_problems(100 if RUN_LONG else 60, seed=999)        # different seed = held-out problems
train_ds = Dataset.from_list([{"prompt": to_prompt(r["question"]), "answer": r["answer"], "question": r["question"]}
                              for r in train_rows])

A greedy evaluation on the held-out problems, so we can compare before and after.

In [ ]:
@torch.no_grad()
def evaluate(model, rows, batch=30):
    model.eval()
    right, fmt = 0, 0
    for i in range(0, len(rows), batch):
        chunk = rows[i:i + batch]
        texts = [tok.apply_chat_template(to_prompt(r["question"]), tokenize=False, add_generation_prompt=True) for r in chunk]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False, pad_token_id=tok.pad_token_id)
        for r, ids in zip(chunk, out[:, enc["input_ids"].shape[1]:]):
            t = tok.decode(ids, skip_special_tokens=True)
            right += extract_int(t) == r["answer"]
            fmt += has_answer_line(t)
    return right / len(rows), fmt / len(rows)

acc_before, fmt_before = evaluate(policy, test_rows)
print(f"BEFORE training: accuracy {acc_before:.0%}, uses 'Answer:' line {fmt_before:.0%}")

The reward functions. TRL calls each one with the batch of `completions` and every extra dataset column (here `answer` and `question`) as keyword arguments, and expects one float per completion.
Our prompts are chat messages, so each completion arrives as a list of messages. The helper handles both shapes.
We also log every reward with its question so we can study the groups afterwards.

In [ ]:
reward_log = []          # (question, total reward) for every completion TRL generates

def _text(c):
    if isinstance(c, list):                                   # conversational format
        return " ".join(m.get("content", "") for m in c if isinstance(m, dict))
    return c or ""

def correct_reward(completions, answer, question=None, **kwargs):
    texts = [_text(c) for c in completions]
    rs = [REWARD_CORRECT if extract_int(t) == int(a) else 0.0 for t, a in zip(texts, answer)]
    fm = [REWARD_FORMAT if has_answer_line(t) else 0.0 for t in texts]
    for q, r, f in zip(question or [""] * len(rs), rs, fm):
        reward_log.append((q, r + f))
    return rs

def format_reward(completions, **kwargs):
    return [REWARD_FORMAT if has_answer_line(_text(c)) else 0.0 for c in completions]

def overlong_reward(completion_ids, **kwargs):
    return [0.5 * overlong_penalty(len(ids), l_max=MAX_NEW, l_cache=32) for ids in completion_ids]

# quick self-test on fake completions, so a typo fails here and not after 10 minutes
fake = [[{"role": "assistant", "content": "7*12=84. Answer: 84"}], [{"role": "assistant", "content": "maybe 80"}]]
print(correct_reward(fake, answer=[84, 84], question=["q", "q"]), format_reward(fake), overlong_reward([[0] * 10, [0] * 90]))
reward_log.clear()

In [ ]:
config = GRPOConfig(
    output_dir="grpo_smollm2",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=16,     # 16 completions per step ...
    num_generations=8,                  # ... = 2 prompts x 8 samples. The batch must divide by num_generations.
    gradient_accumulation_steps=1,
    max_completion_length=MAX_NEW,
    temperature=1.0,
    learning_rate=3e-5,                 # LoRA tolerates a higher rate than full fine-tuning (TRL default is 1e-6)
    beta=0.0,                           # no KL penalty, as in DAPO; also means no reference model in memory
    epsilon=0.2, epsilon_high=0.28,     # DAPO clip-higher
    loss_type="dapo",                   # DAPO token-level loss
    scale_rewards="group",              # divide by the group std, as in the GRPO paper
    bf16=BF16_OK,                       # must be False on a T4
    fp16=HAS_GPU and not BF16_OK,
    gradient_checkpointing=False,       # the model is small; this keeps the PEFT setup simple
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    seed=0,
)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])
trainer = GRPOTrainer(model=policy, reward_funcs=[correct_reward, format_reward, overlong_reward],
                      args=config, train_dataset=train_ds, processing_class=tok, peft_config=lora)
try:
    trainer.model.print_trainable_parameters()
except Exception:
    pass

t0 = time.time()
trainer.train()
train_minutes = (time.time() - t0) / 60
print(f"trained {MAX_STEPS} steps in {train_minutes:.1f} min")

In [ ]:
acc_after, fmt_after = evaluate(trainer.model, test_rows)
print(f"held-out accuracy:   {acc_before:.0%} -> {acc_after:.0%}")
print(f"'Answer:' line rate: {fmt_before:.0%} -> {fmt_after:.0%}")
hist = [h for h in trainer.state.log_history if "reward" in h]
if hist:
    print("mean training reward every 10 steps:", [round(h["reward"], 2) for h in hist])

What to expect: the format reward is learned first (the `Answer:` rate jumps), and accuracy moves a few points. 120 steps of 2 prompts is a very small run. The point is that the whole loop, sample, grade, compare to the group, update, fits on a free GPU.

## 6. Dynamic sampling on the real training data
How much of our training batch was wasted on groups where all 8 answers scored the same? TRL also logs this as `frac_reward_zero_std`.

In [ ]:
from collections import defaultdict
by_q = defaultdict(list)
for q, r in reward_log:
    by_q[q].append(r)
train_groups = [np.array(v) for v in by_q.values() if len(v) >= 2]
zero_var = sum(np.ptp(g) == 0 for g in train_groups)
print(f"{len(train_groups)} groups, {zero_var} ({zero_var / max(1, len(train_groups)):.0%}) had identical rewards -> zero gradient")
print(f"DAPO dynamic sampling would drop them and sample new prompts, keeping {len(dynamic_sampling(train_groups))} useful groups")

In [ ]:
check("STaR produced at least 4 training examples", len(sft) >= 4)
check("GRPO advantages sum to about 0 in each group", all(abs(grpo_advantages(g).sum()) < 1e-6 for g in groups))
check("training ran and logged rewards", len(reward_log) > 0)
check("held-out accuracy did not drop by more than 5 points", acc_after >= acc_before - 0.05)

## Your turn
1. Turn off the format reward (remove `format_reward` and set `REWARD_FORMAT = 0`). Does accuracy still rise, or does the model never learn the `Answer:` line?
2. Change the training problems so they are all very easy (only `+` with one-digit numbers). What happens to the share of zero-variance groups, and why does that matter?
3. Fine-tune SmolLM2 on `star_sft.jsonl` with TRL's `SFTTrainer`. That is the last step of the STaR loop.

**Next:** a06 · Open-ended evolution of self-improving agents

## Solutions

In [ ]:
# Solution 1: correctness reward only. Retrains the model, so it is off by default (~8 minutes).
RUN_NO_FORMAT = False
if RUN_NO_FORMAT:
    p2 = AutoModelForCausalLM.from_pretrained(MID).float().to("cuda" if HAS_GPU else "cpu")
    t2 = GRPOTrainer(model=p2, reward_funcs=[correct_reward], args=config, train_dataset=train_ds,
                     processing_class=tok, peft_config=lora)
    t2.train()
    print("no format reward -> accuracy, format rate:", evaluate(t2.model, test_rows))
# Notice: the fallback parser (last number) still pays for right answers, so accuracy can rise,
# but the Answer: line appears less often. A small shaping reward makes the output easy to grade.

In [ ]:
# Solution 2: easy problems give mostly all-correct groups -> zero gradient.
fake_groups = [np.array([1.2] * 8)] * 15 + [np.array([1.2] * 7 + [0.2])] * 5     # 20 easy prompts, 8 samples each
print(f"if 15 of 20 easy groups are all correct, dynamic sampling keeps {len(dynamic_sampling(fake_groups))} of 20")
# Notice: RL learns only where the model is sometimes right and sometimes wrong. Pick training
# problems near the edge of what the model can do. This is why DAPO's dynamic sampling helps.

In [ ]:
# Solution 3: the last STaR step, supervised fine-tuning on the kept rationales (~2 minutes on a T4).
RUN_STAR_SFT = False
if RUN_STAR_SFT:
    from trl import SFTConfig, SFTTrainer
    sft_ds = Dataset.from_list([{"messages": r["messages"]} for r in sft])
    base = AutoModelForCausalLM.from_pretrained(MID).float().to("cuda" if HAS_GPU else "cpu")
    st = SFTTrainer(model=base, train_dataset=sft_ds, processing_class=tok, peft_config=lora,
                    args=SFTConfig(output_dir="star_sft", num_train_epochs=3, per_device_train_batch_size=4,
                                   learning_rate=1e-4, bf16=BF16_OK, fp16=HAS_GPU and not BF16_OK,
                                   save_strategy="no", report_to="none"))
    st.train()
    print("after STaR SFT -> accuracy, format rate:", evaluate(st.model, test_rows))
# Notice: STaR restarts from the BASE model each outer loop and trains on the growing dataset,
# rather than training the same model again and again. That limits drift.